In [1]:
# MSPB D1 preprocessing 

import pandas as pd
import numpy as np
from pathlib import Path
import json

In [2]:
# Paths

DATA_PATH = "../backend/data/D1_sensor_data.csv"
OUT_PATH  = "../backend/data/D1_clean_all_hives_15min.parquet"

QC_TABLE_PATH      = "../backend/data/hive_qc_table.csv"
DROPPED_HIVES_PATH = "../backend/data/dropped_hives.csv"
SUMMARY_PATH       = "../backend/data/final_dataset_summary.csv"
DT_SUMMARY_PATH    = "../backend/data/final_dataset_dt_summary.csv"

PROTO_OUT = Path("..") / "backend" / "data" / "mspb_clean_prototype.csv"


In [3]:
# Settings

ROUND_RULE = "15min"
EXPECTED_SAMPLES_PER_DAY = 96  # 24*60 / 15

# thresholds to remove broken hives
MIN_UNIQUE_TS = 500
MIN_MED_SAMPLES_PER_DAY = 60
MIN_COVERAGE = 0.50

HIVE_POWER_SENTINEL = -10.0
HIVE_POWER_EPS = 1e-6

TEMP_MIN, TEMP_MAX = -10.0, 60.0
HUM_MIN, HUM_MAX   = 0.0, 100.0

ROLL_STEPS = 4  # 4 * 15min = last 1 hour (causal)

EXPORT_MULTIMODAL_PROTO = True
PROTO_TAIL_ROWS = 2000


In [4]:
# Helpers

def canonical_hive_id(x) -> str:

    # Normalize hive IDs to avoid QC index mismatches
    if pd.isna(x):
        return ""
    s = str(x).strip()
    try:
        v = float(s)
        if np.isfinite(v) and abs(v - round(v)) < 1e-9:
            return str(int(round(v)))
        return s
    except Exception:
        return s


def hz_cols_from(df: pd.DataFrame) -> list[str]:
    
# Find hz_* columns and sort them by frequency where possible.

    hz = [c for c in df.columns if c.startswith("hz_")]
    def keyfunc(c):
        try:
            return float(c.replace("hz_", ""))
        except Exception:
            return float("inf")
    # stable: numeric first, then string order
    return sorted(hz, key=lambda c: (keyfunc(c), c))


def robust_agg(df: pd.DataFrame, hz_cols: list[str]) -> pd.DataFrame:
    """
    Aggregate collisions inside each (tag_number, t_bin).
      - numeric: median
      - dup_count: sum
      - flags: max
      - within-bin std: max (worst conflict in that bin)
    """
    agg_map = {
        "temperature": "median",
        "humidity": "median",
        "hive_power": "median",
        "audio_density": "median",
        "audio_density_ratio": "median",
        "density_variation": "median",

        "dup_count": "sum",
        "geo_missing_flag": "max",
        "hive_power_missing_flag": "max",

        "temperature_bin_std": "max",
        "humidity_bin_std": "max",
        "hive_power_bin_std": "max",
        "audio_density_bin_std": "max",
    }
    for c in hz_cols:
        agg_map[c] = "median"

    return df.groupby(["tag_number", "t_bin"], as_index=False).agg(agg_map)


def build_audio_features(df: pd.DataFrame, hz_cols: list[str]) -> pd.DataFrame:

#Compress audio spectrum into simple, interpretable features.

    if not hz_cols:
        return df

    df = df.copy()
    df["audio_spectral_mean_db"] = df[hz_cols].mean(axis=1)
    df["audio_spectral_std_db"]  = df[hz_cols].std(axis=1)


    # compute freqs from column names where possible
    freqs = np.array([float(c.replace("hz_", "")) if _is_float(c.replace("hz_", "")) else np.nan for c in hz_cols], dtype=float)
    
    # replace NaNs in freqs with evenly spaced indices 
    if np.isnan(freqs).any():
        freqs = np.arange(len(hz_cols), dtype=float)

    x = freqs - freqs.mean()
    denom = np.sum(x**2)

    y = df[hz_cols].to_numpy(dtype=float)
    y_center = y - np.nanmean(y, axis=1, keepdims=True)
    df["audio_spectral_slope"] = (y_center @ x) / denom if denom != 0 else 0.0

    return df.drop(columns=hz_cols)


def _is_float(s: str) -> bool:
    try:
        float(s)
        return True
    except Exception:
        return False


def add_dt_prev_minutes(df_15: pd.DataFrame) -> pd.DataFrame:
    df_15 = df_15.sort_values(["tag_number", "published_at"]).copy()
    df_15["dt_prev_min"] = (
        df_15.groupby("tag_number")["published_at"]
             .diff()
             .dt.total_seconds() / 60.0
    )
    return df_15


def add_causal_roll_features(df_15: pd.DataFrame, cols: list[str], roll_steps: int) -> pd.DataFrame:
    """
    GUARANTEED per-hive causal rolling (no leakage across hives):
      rolling_mean(t) uses only [t-1, t-2, ...] within SAME hive.
    """
    df_15 = df_15.sort_values(["tag_number", "published_at"]).copy()

    for col in cols:
        if col in df_15.columns:
            feat = f"{col}_roll_mean_{roll_steps}"
            df_15[feat] = (
                df_15.groupby("tag_number")[col]
                     .transform(lambda s: s.shift(1).rolling(roll_steps, min_periods=1).mean())
            )

            # Leakage guard: first row per hive has no past -> must be NaN
            first_idx = df_15.groupby("tag_number").head(1).index
            assert df_15.loc[first_idx, feat].isna().all(), \
                f"{feat} has values at first timestep per hive => leakage risk (should use shift(1))"

    return df_15


In [5]:
# FULL GRID + MISSINGNESS

def add_full_15min_grid_and_missingness(df_15: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
   Create a full 15-min time grid per hive and measure missing data.
   Returns the full grid and per-hive missingness stats.
    """
    out = []
    qc_extra = []

    for hive_id, g in df_15.groupby("tag_number"):
        g = g.sort_values("published_at").copy()

        # align start/end to 15-min grid
        start = g["published_at"].min().floor(ROUND_RULE)
        end   = g["published_at"].max().ceil(ROUND_RULE)

        full_index = pd.date_range(
            start=start,
            end=end,
            freq=ROUND_RULE,
            tz="UTC"
        )

        full = pd.DataFrame({"published_at": full_index})
        full["tag_number"] = hive_id

        full = full.merge(g, on=["tag_number", "published_at"], how="left")

        # Missing bin means: there was no original data in that bin
        full["missing_bin_flag"] = full["dup_count"].isna().astype(int)

        # Fill dup_count for downstream safety
        full["dup_count"] = full["dup_count"].fillna(0)

        pct_missing_bins = float(full["missing_bin_flag"].mean())

        # Longest consecutive missing stretch
        is_missing = full["missing_bin_flag"].to_numpy()
        longest = 0
        cur = 0
        for v in is_missing:
            if v == 1:
                cur += 1
                longest = max(longest, cur)
            else:
                cur = 0

        qc_extra.append({
            "tag_number": hive_id,
            "pct_missing_bins": pct_missing_bins,
            "longest_missing_stretch_bins": int(longest),
            "longest_missing_stretch_hours": float(longest) * 0.25,
            "missing_rate_temperature": float(full["temperature"].isna().mean()) if "temperature" in full.columns else np.nan,
            "missing_rate_humidity": float(full["humidity"].isna().mean()) if "humidity" in full.columns else np.nan,
            "missing_rate_audio": float(full["audio_density"].isna().mean()) if "audio_density" in full.columns else np.nan,
            "missing_rate_hive_power": float(full["hive_power"].isna().mean()) if "hive_power" in full.columns else np.nan,
            "mean_conflict_score": float(full["conflict_score"].mean()) if "conflict_score" in full.columns else np.nan,
        })

        out.append(full)

    df_full = pd.concat(out, ignore_index=True)
    qc_missing = pd.DataFrame(qc_extra).set_index("tag_number")

    return df_full, qc_missing

In [6]:
# QC using observed bins only

def compute_hive_qc_observed_only(df_full: pd.DataFrame):
    """
    QC metrics computed ONLY on observed bins (missing_bin_flag==0):
      - unique_timestamps
      - median_samples_per_day
      - coverage = median_samples_per_day / 96
    Missingness metrics are computed separately from full grid and joined later.
    """
    tmp = df_full.copy()
    tmp["date_utc"] = tmp["published_at"].dt.floor("D")

    observed = tmp[tmp["missing_bin_flag"] == 0]

    # If a hive has 0 observed bins, it will just drop out here
    spd = (
        observed.groupby(["tag_number", "date_utc"])["published_at"]
                .nunique()
                .reset_index(name="samples_per_day")
    )

    med_spd   = spd.groupby("tag_number")["samples_per_day"].median() if len(spd) else pd.Series(dtype=float)
    unique_ts = observed.groupby("tag_number")["published_at"].nunique() if len(observed) else pd.Series(dtype=int)

    qc = pd.DataFrame({
        "unique_timestamps": unique_ts,
        "median_samples_per_day": med_spd
    }).fillna(0)

    qc["coverage"] = qc["median_samples_per_day"] / EXPECTED_SAMPLES_PER_DAY

    keep = qc[
        (qc["unique_timestamps"] >= MIN_UNIQUE_TS) &
        (qc["median_samples_per_day"] >= MIN_MED_SAMPLES_PER_DAY) &
        (qc["coverage"] >= MIN_COVERAGE)
    ].index

    return qc, keep


In [7]:
# Load + parse time

print("Loading:", DATA_PATH)
df = pd.read_csv(DATA_PATH)

# required columns check 
required = ["published_at", "tag_number", "temperature", "humidity", "hive_power",
            "audio_density", "audio_density_ratio", "density_variation", "lat", "long"]
missing_required = [c for c in required if c not in df.columns]
assert not missing_required, f"Missing required columns in raw CSV: {missing_required}"

df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
df = df.dropna(subset=["published_at"]).copy()

hz_cols = hz_cols_from(df)

keep_cols = [
    "published_at", "tag_number",
    "temperature", "humidity", "hive_power",
    "audio_density", "audio_density_ratio", "density_variation",
    "lat", "long"
] + hz_cols

df = df[keep_cols].copy()
print("Loaded rows:", len(df), "cols:", df.shape[1], "hives:", df["tag_number"].nunique())


Loading: ../backend/data/D1_sensor_data.csv
Loaded rows: 960809 cols: 26 hives: 85


In [8]:
# Sentinel / invalid -> NaN (and flags)

df["geo_missing_flag"] = ((df["lat"] == 0) & (df["long"] == 0)).astype(int)
df.loc[df["geo_missing_flag"] == 1, ["lat", "long"]] = np.nan

df.loc[~df["temperature"].between(TEMP_MIN, TEMP_MAX), "temperature"] = np.nan
df.loc[~df["humidity"].between(HUM_MIN, HUM_MAX), "humidity"] = np.nan

df.loc[np.isclose(df["hive_power"], HIVE_POWER_SENTINEL, atol=HIVE_POWER_EPS), "hive_power"] = np.nan
df["hive_power_missing_flag"] = df["hive_power"].isna().astype(int)


In [9]:
# Floor timestamps to 15-min bins + collision tracking

df["t_bin"] = df["published_at"].dt.floor(ROUND_RULE)
df["dup_count"] = 1

bin_keys = ["tag_number", "t_bin"]
for col in ["temperature", "humidity", "hive_power", "audio_density"]:
    if col in df.columns:
        df[f"{col}_bin_std"] = df.groupby(bin_keys)[col].transform("std")

In [10]:
# Aggregate to 15-min per hive

df_15 = robust_agg(df, hz_cols=hz_cols)
df_15 = df_15.rename(columns={"t_bin": "published_at"})

std_cols = ["temperature_bin_std", "humidity_bin_std", "hive_power_bin_std", "audio_density_bin_std"]
for c in std_cols:
    if c in df_15.columns:
        df_15[c] = df_15[c].fillna(0.0)

df_15 = build_audio_features(df_15, hz_cols=hz_cols)

df_15["conflict_score"] = (
    df_15["dup_count"].clip(upper=10).astype(float)
    + df_15.get("temperature_bin_std", 0.0)
    + df_15.get("humidity_bin_std", 0.0)
    + df_15.get("hive_power_bin_std", 0.0)
    + df_15.get("audio_density_bin_std", 0.0)
)

# Build FULL GRID

df_full, qc_missing = add_full_15min_grid_and_missingness(df_15)


In [ ]:
# QC on observed bins only + join missingness

qc_table, keep_hives = compute_hive_qc_observed_only(df_full)
qc_table = qc_table.join(qc_missing, how="left")
print("QC note: coverage/unique_timestamps/median_samples_per_day computed on OBSERVED bins only (missing_bin_flag==0).")

#  unique timestamps must match observed-only counts
_check = df_full[df_full["missing_bin_flag"] == 0].groupby("tag_number")["published_at"].nunique()
aligned = _check.reindex(qc_table.index).fillna(0).astype(int)
assert np.allclose(aligned.values, qc_table["unique_timestamps"].fillna(0).astype(int).values), \
    "QC unique_timestamps is not observed-only. Check compute_hive_qc_observed_only()."

# normalize hive ids for safe indexing + prototype selection
qc_table = qc_table.copy()
qc_table.index = qc_table.index.map(canonical_hive_id)

qc_table.to_csv(QC_TABLE_PATH)

keep_hives_norm = set([canonical_hive_id(h) for h in keep_hives])
dropped_hives = sorted(set(qc_table.index) - keep_hives_norm)
pd.DataFrame({"dropped_hives": dropped_hives}).to_csv(DROPPED_HIVES_PATH, index=False)

print(f"Kept hives: {len(keep_hives_norm)} out of {df_15['tag_number'].nunique()}")
print("Saved QC table ->", QC_TABLE_PATH)
print("Saved dropped hives ->", DROPPED_HIVES_PATH)

# filter to kept hives on FULL GRID dataset
df_full["tag_number"] = df_full["tag_number"].map(canonical_hive_id)
df_15_final = df_full[df_full["tag_number"].isin(keep_hives_norm)].copy()

QC note: coverage/unique_timestamps/median_samples_per_day computed on OBSERVED bins only (missing_bin_flag==0).
Kept hives: 52 out of 85
Saved QC table -> ../backend/data/hive_qc_table.csv
Saved dropped hives -> ../backend/data/dropped_hives.csv


In [12]:
# Missingness presence flags

df_15_final["has_temp"]  = df_15_final["temperature"].notna().astype(int)
df_15_final["has_humid"] = df_15_final["humidity"].notna().astype(int)
df_15_final["has_audio"] = df_15_final["audio_density"].notna().astype(int)
df_15_final["has_power"] = df_15_final["hive_power"].notna().astype(int)

In [13]:
# Gap features + rolling (on full grid)

df_15_final = add_dt_prev_minutes(df_15_final)

# first bin per hive has no previous timestamp → treat as expected 15 minutes
df_15_final["dt_prev_min"] = df_15_final["dt_prev_min"].fillna(15)

df_15_final["gap_30min_flag"] = (df_15_final["dt_prev_min"] > 30).astype(int)
df_15_final["gap_2hr_flag"]   = (df_15_final["dt_prev_min"] > 120).astype(int)

pd.DataFrame([{
    "median_dt_prev_min": float(df_15_final["dt_prev_min"].median(skipna=True)),
    "pct_gap_30min": float(df_15_final["gap_30min_flag"].mean() * 100),
    "pct_gap_2hr": float(df_15_final["gap_2hr_flag"].mean() * 100),
}]).to_csv(DT_SUMMARY_PATH, index=False)
print("Saved dt summary ->", DT_SUMMARY_PATH)

roll_cols = ["temperature", "humidity", "audio_density", "hive_power"]
df_15_final = add_causal_roll_features(df_15_final, cols=roll_cols, roll_steps=ROLL_STEPS)

Saved dt summary -> ../backend/data/final_dataset_dt_summary.csv


In [14]:
# Final sort + save dataset

df_15_final = df_15_final.sort_values(["tag_number", "published_at"]).reset_index(drop=True)


assert not df_15_final.duplicated(subset=["tag_number", "published_at"]).any(), "Duplicate keys in final dataset!"
# check 15-min grid alignment
assert (df_15_final["published_at"].dt.minute % 15 == 0).all(), "Some timestamps are not aligned to 15-min grid!"
# monotonic within hive
mono_ok = df_15_final.groupby("tag_number")["published_at"].apply(lambda s: s.is_monotonic_increasing).all()
assert mono_ok, "Timestamps not monotonic within a hive after final sort!"

summary = {
    "rows_after_filter": int(len(df_15_final)),
    "kept_hives": int(df_15_final["tag_number"].nunique()),
    "start_time": str(df_15_final["published_at"].min()),
    "end_time": str(df_15_final["published_at"].max()),
}
pd.DataFrame([summary]).to_csv(SUMMARY_PATH, index=False)
print("Saved final dataset summary ->", SUMMARY_PATH)


Saved final dataset summary -> ../backend/data/final_dataset_summary.csv


In [15]:
# Export data schema JSON 

SCHEMA_PATH = "../backend/data/data_schema.json"

schema_desc = {
    "published_at": "UTC timestamp, 15-min grid per hive (full grid).",
    "tag_number": "Hive identifier (normalized).",
    "missing_bin_flag": "1 if the 15-min bin had no original data, else 0.",
    "dup_count": "How many raw rows were aggregated into this bin.",
    "conflict_score": "Collision/conflict measure (dup_count + within-bin std).",
    "dt_prev_min": "Minutes since previous bin within same hive (used for gap/noise scaling).",
    "gap_30min_flag": "1 if dt_prev_min > 30 minutes.",
    "gap_2hr_flag": "1 if dt_prev_min > 120 minutes.",
    "has_temp": "1 if temperature exists in this bin.",
    "has_humid": "1 if humidity exists in this bin.",
    "has_audio": "1 if audio_density exists in this bin.",
    "has_power": "1 if hive_power exists in this bin.",
}

schema = []
for col in df_15_final.columns:
    schema.append({
        "name": col,
        "dtype": str(df_15_final[col].dtype),
        "description": schema_desc.get(col, "")
    })

with open(SCHEMA_PATH, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2)

print("Saved data schema ->", SCHEMA_PATH)

try:
    df_15_final.to_parquet(OUT_PATH, index=False)
    print("Saved Parquet:", OUT_PATH)
except ImportError:
    csv_path = OUT_PATH.replace(".parquet", ".csv")
    df_15_final.to_csv(csv_path, index=False)
    print("pyarrow not installed -> saved CSV instead:", csv_path)

Saved data schema -> ../backend/data/data_schema.json
Saved Parquet: ../backend/data/D1_clean_all_hives_15min.parquet
